In [2]:
from langchain_community.document_loaders import PyPDFLoader

C:\Users\Abishek M\AppData\Local\Temp\ipykernel_25700\4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\Abishek M\Desktop\Projects\assignment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load first PDF
loader1 = PyPDFLoader("API Documentation Partial.pdf")
documents = loader1.load()

In [3]:
num_pages = len(documents)

# Combine all text
full_text = " ".join([doc.page_content for doc in documents])

# Total words
total_words = len(full_text.split())

# Total characters
total_characters = len(full_text)

print(f"Total Pages: {num_pages}")
print(f"Total Words: {total_words}")
print(f"Total Characters: {total_characters}")

Total Pages: 72
Total Words: 9892
Total Characters: 100998


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True}
)

C:\Users\Abishek M\AppData\Local\Temp\ipykernel_19192\2696727994.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9212.35it/s]


In [6]:
embeddings = embedding_model.embed_documents(
    [chunk.page_content for chunk in chunks]
)
print("Number of embeddings:", len(embeddings))
print("Embedding dimension:", len(embeddings[0]))


Number of embeddings: 273
Embedding dimension: 384


In [8]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embedding_model
)
vectorstore.save_local("DB/faiss_index")

In [9]:
def semantic_search(user_query, vectorstore, k=3):

    results = vectorstore.similarity_search(
        user_query,
        k=k
    )

    return [doc.page_content for doc in results]

In [10]:
from huggingface_hub import InferenceClient
from dotenv import load_dotenv
import os

load_dotenv()

from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    api_key=os.getenv("API_KEY"),
    base_url="https://api.deepinfra.com/v1/openai"
)

In [20]:
def generate_rag_answer(query, vectorstore):

    # ==========================================
    # RETRIEVE RELEVANT CHUNKS
    # ==========================================

    retrieved_docs = semantic_search(
        query,
        vectorstore,
        k=3
    )

    # ==========================================
    # BUILD CONTEXT
    # ==========================================

    context = "\n\n".join(retrieved_docs)

    # ==========================================
    # SYSTEM PROMPT
    # ==========================================

    system_prompt = """
You are a Senior Upwork API Consultant.

Rules:
1. Answer ONLY from the provided context.
2. Do NOT hallucinate or make up information.
3. If the answer is not available in the context,
   respond with:
   "I'm sorry, but the provided documentation does not contain that information."
4. Keep answers concise and technical.
"""

    # ==========================================
    # USER PROMPT
    # ==========================================

    user_prompt = f"""
Context:
{context}

Question:
{query}
"""

    # ==========================================
    # GENERATE RESPONSE
    # ==========================================

    response = client.chat.completions.create(
        model=os.getenv("MODEL_NAME"),

        messages=[

            {
                "role": "system",
                "content": system_prompt
            },

            {
                "role": "user",
                "content": user_prompt
            }

        ],

        max_tokens=300,
        temperature=0.3
    )

    # ==========================================
    # RETURN FINAL ANSWER
    # ==========================================

    return response.choices[0].message.content

In [22]:
print(generate_rag_answer("How long is an OAuth access token valid for?",vectorstore))

According to the documentation, an OAuth access token is valid for 24 hours (86400 seconds).
